Tässä demossa tutustutaan datan ryhmittelyyn ja yhdistämiseen. Lisäksi tehdään frekvenssi- ja pivottaulukoita sekä piirretään
aiheeseen sopivia kaavioita. 

Demossa käytetään Seaborn-kirjastossa olevaa demodataa diamonds. 

Sarakkeet x, y ja z ovat timantin mittoja millimetreinä. Hinta ilmoitetaan dollareina. 

In [ ]:
import seaborn as sns

df = sns.load_dataset("diamonds")
df.head()

Tulostetaan seuraavaksi sarakkeen cut arvoja. Tutki komentoja poistamalla kommenttimerkki ja ajamalla solu. 

In [ ]:

#df["cut"].unique() #Uniikit arvot. 

#sorted(df["cut"].unique())#Uniikit arvot järjestettynä. 
#sorted(df["cut"].unique(), reverse=True) #Uniikit arvot suurimmasta pienimpään. Eli tässä tapauksessa öökkösjärjestyksessä.

#df["cut"].nunique() #Uniikkien arvojen lukumäärä.
#df["cut"].value_counts() #Uniikit arvot ja niiden esiintymien lukumäärä. 

Tehdään seuraavaksi frekvenssitaulukko, jossa on täsmälleen samat tiedot kuin yllä olevassa tulostuksessa. Käytetään samaa komentoa, 
mutta sijoitetaan tulos dataframeen. Muutetaan järjestys siten, että huonoin arvo on ylimpänä (.sort_index(ascending = False)). Nimetään lisäksi sarake count uudelleen. Inplace = True -argumentti kertoo, että muutos tehdään olemassa olevaan dataframeen. 

In [ ]:
df_cut = df["cut"].value_counts().sort_index(ascending = False).reset_index()

df_cut.rename(columns={"count": "frequency"}, inplace=True)

df_cut

Lisätään seuraavaksi sarake, jossa lasketaan kumulatiivinen frekvenssi. 

In [ ]:
df_cut["cumulative frequency"] = df_cut["frequency"].cumsum()

df_cut

Lasketaan seuraavaksi suhteelliset frekvenssit sekä kumulatiiviset suhteelliset frekvenssit. Ilmoitetaan arvot suoraan
prosentteina. 

In [ ]:
df_cut["relative frequency(%)"] = (
    df_cut["frequency"] / df_cut["frequency"].sum() * 100
)

df_cut

In [ ]:
df_cut["cumulative relative frequency(%)"] = (
    df_cut["relative frequency(%)"].cumsum()
)

df_cut

Lisätään seuraavaksi dataframeen summarivit. 



In [ ]:
i = len(df_cut)

df_cut.loc[i, "frequency"] = df_cut["frequency"].sum()
df_cut.loc[i, "relative frequency(%)"] = 100.0

Pyöristetään lopuksi tarvittavien sarakkeiden arvot yhden desimaalin tarkkuuteen. Tämän olisi voinut
tehdä jo laskennan yhteydessä. 

In [ ]:
df_cut["relative frequency(%)"] = df_cut["relative frequency(%)"].round(1)
df_cut["cumulative relative frequency(%)"] = df_cut["cumulative relative frequency(%)"].round(1)

df_cut


Tulostetaan lopuksi taulukko siten, että NaN-arvot korvataan tyhjällä. Vaikka aikaisemmin sarakkaiden arvot 
pyöristettiin yhden desimaalin tarkkuuteen, täytyy nyt kertoa, että tulostettaessa näytetään vain yksi desimaali. 

In [ ]:


df_cut.style.format(
    {
        "frequency": "{:.0f}",
        "cumulative frequency": "{:.0f}",
        "relative frequency(%)": "{:.1f}",
        "cumulative relative frequency(%)": "{:.1f}"
    },
    na_rep=""
)

Tehdään seuraavaksi ristiintaulukointi leikkauksen ja värin sekä leikkauksen ja puhtauden suhteen. 

In [ ]:
import pandas as pd

pd.crosstab(df["cut"], df["color"])

In [ ]:
pd.crosstab(df["cut"], df["clarity"])

Seuraavaksi lasketaan värien prosenttiosuudet kussakin leikkausluokassa. Normalize = "index" tarkoittaa, että prosenttiosuudet
lasketaan riveittäin. Mikäli prosenttiosuudet haluttaisiin sarakkeittain, käytettäisiin komentoa normalize = "columns". Kokeile sitä
ja mieti, mitä taulukko silloin esittää. Kokeile myös vaihtaa rivit ja sarakkeet toisin päin (vaihdat järjestystä koodissa). 

Muotoilut sivuutetaan tässä. 

In [ ]:
pd.crosstab(
    df["cut"],
    df["color"],
    normalize="index"
) * 100

Ryhmitellään seuraavaksi aineisto leikkauksen mukaan ja lasketaan kunkin ryhmän hinnan keskiarvo. Sarakkeen cut arvot ovat ns. 
kategorisia muuttujia. Komennolla observed = True ryhmittelyyn otetaan mukaan vain sellaiset arvot, jotka esiintyvät datassa. 
Kategorinen muuttuja tarkoittaa joukkoa etukäteen asetettuja sallittuja arvoja. 

In [ ]:
df.groupby("cut", observed=True)["price"].mean()

Ryhmitellään seuraavaksi data leikkauslaadun ja värin mukaan. Ryhmistä lasketaan hintojen keskiarvo. Komennolla Python muodostaa kaikki mahdolliset ryhmät. Otetaan mukaan vain datassa esiintyvät ryhmät. 

In [ ]:
df.groupby(["cut", "color"], observed=True)["price"].mean()

Mieti, mitä alla oleva koodi tekee.

In [ ]:
df.groupby(
    ["cut", "color", "clarity"],
    observed=True
)["price"].max()

Lasketaan seuraavaksi ryhmittäin tunnuslukuja. Komennon .agg avulla voidaan laskea useita tunnuslukuja yhtäaikaa.

In [ ]:
df.groupby(
    ["cut", "color"],
    observed=True
)["price"].agg(["count", "mean", "median"])

Tehdään Pivot-taulukko, jossa lasketaan keskihinnat leikkauksen ja värin mukaan. Käytettäviä funktioita ovat mm. count, sum,
mean, median, min, max ja std. Useiden funktioiden käyttäminen yhtäaikaa on myös mahdollista aggfunc = ["count","max"].

In [ ]:
df.pivot_table(
    values="price",
    index="cut",
    columns="color",
    aggfunc="mean",
    observed=True
)

Tehdään aikaisemmin tehty frekvenssitaulukko pivot-toiminnalla. Huomaa, että tässä ei ole mahdollista laskea kumulatiivisia ja 
suhteellisia frekvenssejä. 

In [ ]:
df.pivot_table(
    values="price",
    index="cut",
    #columns="cut",
    aggfunc="count",
    observed=True
)

Tehdään lopuksi timattien hinnoista frekvenssitaulukko käyttäen yhdistettyjä luokkia. Aluksi määritellään luokkarajat. Viimeisessä luokassa on oikaistu. Luokkien pitäisi olla saman pituisia. Ensimmäinen ja viimeinen luokka voivat olla ns. avoimia. 

Kiinnitä huomiota komentoon right = False. Tämä tekee luokista vasemmalta suljettuja, jolloin binsiin annettu arvo on kunkin luokan alaraja: [0,1000[, 
[1000,2000[ jne. 



In [ ]:
bins = [0, 1000, 2000, 3000, 4000, 5000, 10000, 20000]

labels = [
    "less than 1000",
    "1000–1999",
    "2000–2999",
    "3000–3999",
    "4000–4999",
    "5000–9999",
    "10000–19999"
]

df["price class"] = pd.cut(
    df["price"],
    bins=bins,
    labels=labels,
    right=False
)



In [ ]:
df.pivot_table(
    values="price",
    index="price class",
    #columns="cut",
    aggfunc="count",
    observed=True
)

